# Phase 4: Mini RAG — Retrieval-Augmented Generation from Scratch

**Goal:** Build a complete RAG pipeline using only numpy + SBERT + a free HuggingFace model.
No LangChain, no Pinecone — every piece built by hand so you understand it deeply.

**Pipeline you'll build:**
```
Document
  └─► Chunker           → overlapping text chunks
        └─► Embedder    → SBERT vectors per chunk
              └─► Index → numpy vector store

Query
  └─► Embedder          → query vector
        └─► Retriever   → cosine similarity → top-k chunks
              └─► Generator (Flan-T5) → final answer
```

**Model used for generation:** `google/flan-t5-small` — free, no API key needed, runs on CPU.

## Step 1: Install & Import

In [ ]:
# Install (run once)
# !pip install sentence-transformers transformers torch

In [ ]:
import numpy as np
import json
import textwrap
from sentence_transformers import SentenceTransformer, util
from transformers import T5ForConditionalGeneration, T5Tokenizer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Step 2: The Document (Knowledge Base)

In a real RAG system this would be your PDFs, docs, etc.
Here we use a self-contained text about transformers and BERT — topics we know well.

In [ ]:
DOCUMENT = """
BERT: Bidirectional Encoder Representations from Transformers

BERT is a transformer-based language model developed by Google AI in 2018. Unlike earlier models
that processed text left-to-right or right-to-left, BERT reads the entire sequence of words at once,
allowing it to understand context from both directions simultaneously. This bidirectional approach
is what makes BERT especially powerful for understanding language.

Pre-training Tasks

BERT is trained using two main tasks. The first is Masked Language Modeling (MLM), where random
words in a sentence are replaced with a [MASK] token, and the model learns to predict the original
word. For example, given 'The cat [MASK] on the mat', BERT should predict 'sat'. This forces the
model to develop a deep understanding of context from both directions.

The second pre-training task is Next Sentence Prediction (NSP). Given two sentences, BERT learns
to predict whether the second sentence naturally follows the first in the original document. This
helps BERT understand relationships between sentences, which is important for question answering
and natural language inference tasks.

Architecture Details

BERT-base has 12 transformer layers, 768 hidden dimensions, and 12 attention heads, totaling
about 110 million parameters. BERT-large has 24 layers, 1024 hidden dimensions, and 16 attention
heads, totaling 340 million parameters. Each input is represented as a combination of three
embeddings: the token embedding (the word itself), the segment embedding (which sentence it
belongs to), and the position embedding (where in the sequence it appears).

Fine-tuning and Applications

After pre-training, BERT can be fine-tuned on downstream tasks with a small amount of task-specific
data. This is called the pre-train then fine-tune paradigm. BERT can be fine-tuned for text
classification by adding a linear layer on top of the [CLS] token. For question answering, it
can predict the start and end positions of the answer within a passage. For named entity recognition,
it can classify each token as a person, organization, location, or other entity.

Sentence-BERT (SBERT)

Standard BERT is not ideal for semantic similarity tasks because comparing two sentences requires
running both through the model together, which is computationally expensive. Sentence-BERT (SBERT)
addresses this by fine-tuning BERT with a siamese network structure. Two sentences are processed
independently, and the model is trained so that semantically similar sentences produce similar
embedding vectors. SBERT reduces the time to find similar sentences in a large corpus from hours
to milliseconds.

Retrieval-Augmented Generation (RAG)

RAG is a technique that combines a retrieval system with a language model generator. First, a
document corpus is split into chunks and each chunk is embedded using a model like SBERT. These
embeddings are stored in a vector database. When a user asks a question, the question is also
embedded, and the vector database is searched for the most similar chunks using cosine similarity.
The retrieved chunks are then included in the prompt sent to a language model, which generates
a final answer grounded in the retrieved context.

Vector Databases and Cosine Similarity

A vector database stores high-dimensional embedding vectors and supports fast nearest-neighbor
search. Popular options include FAISS, Pinecone, Weaviate, and Chroma. The most common similarity
metric is cosine similarity, which measures the cosine of the angle between two vectors. A score
of 1.0 means the vectors point in the same direction (identical meaning), 0.0 means they are
orthogonal (unrelated), and negative values indicate opposite meaning.

Word2Vec and Static Embeddings

Before BERT, Word2Vec was a widely used method for generating word embeddings. It learns word
vectors by training a shallow neural network to predict context words given a target word (Skip-gram)
or predict a target word given context words (CBOW). The resulting vectors capture semantic
relationships: words used in similar contexts get similar vectors. However, Word2Vec produces
static embeddings — each word has a single fixed vector regardless of context. The word 'bank'
gets the same vector whether it refers to a river bank or a financial institution.
"""

print(f'Document length: {len(DOCUMENT)} characters, ~{len(DOCUMENT.split())} words')

## Step 3: The Chunker

Split the document into overlapping chunks. Overlap ensures that context at chunk boundaries
isn't lost — a sentence that straddles a boundary will appear in both adjacent chunks.

In [ ]:
def chunk_text(text, chunk_size=150, overlap=30):
    """
    Split text into overlapping chunks of roughly chunk_size words.
    
    Args:
        text: raw document text
        chunk_size: target number of words per chunk
        overlap: number of words to repeat between consecutive chunks
    
    Returns:
        List of text chunk strings
    """
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk.strip())
        
        if end == len(words):
            break
        start += chunk_size - overlap  # Slide forward, keeping 'overlap' words
    
    return chunks


chunks = chunk_text(DOCUMENT, chunk_size=120, overlap=25)

print(f'Document split into {len(chunks)} chunks')
print()
for i, chunk in enumerate(chunks):
    print(f'--- Chunk {i+1} ({len(chunk.split())} words) ---')
    print(textwrap.fill(chunk, width=90))
    print()

## Step 4: The Embedder — Encode All Chunks

In [ ]:
print('Loading embedding model (SBERT)...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print('Encoding chunks...')
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

print(f'\nChunk embeddings shape: {chunk_embeddings.shape}')
print(f'  = {chunk_embeddings.shape[0]} chunks × {chunk_embeddings.shape[1]} dimensions')

## Step 5: The Vector Store (numpy index)

In production this would be FAISS or Pinecone. Here we build it from scratch
using numpy so you see exactly what's happening.

In [ ]:
class NumpyVectorStore:
    """
    A simple in-memory vector store backed by numpy.
    Supports adding vectors and searching by cosine similarity.
    """
    def __init__(self):
        self.embeddings = None   # shape: [n_docs, dim]
        self.documents  = []     # the raw text chunks
        self.metadata   = []     # optional metadata per chunk

    def add(self, documents, embeddings, metadata=None):
        """Add documents and their pre-computed embeddings."""
        self.documents = list(documents)
        self.embeddings = np.array(embeddings)
        # Normalize all vectors to unit length for fast cosine via dot product
        norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        self.embeddings = self.embeddings / (norms + 1e-10)
        self.metadata = metadata or [{} for _ in documents]
        print(f'Vector store: {len(self.documents)} documents, {self.embeddings.shape[1]}d vectors')

    def search(self, query_embedding, top_k=3):
        """
        Find top-k most similar documents using cosine similarity.
        Since vectors are normalized, cosine similarity = dot product.
        """
        # Normalize query vector
        q = np.array(query_embedding)
        q = q / (np.linalg.norm(q) + 1e-10)

        # Dot product with all stored vectors = cosine similarity
        scores = self.embeddings @ q   # shape: [n_docs]

        # Get top-k indices
        top_indices = np.argsort(scores)[::-1][:top_k]

        return [
            {
                'text': self.documents[i],
                'score': float(scores[i]),
                'chunk_id': int(i),
            }
            for i in top_indices
        ]

    def __len__(self):
        return len(self.documents)


# Build the index
vector_store = NumpyVectorStore()
vector_store.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    metadata=[{'chunk_id': i} for i in range(len(chunks))]
)

print('Vector store ready!')

## Step 6: The Retriever — Find Relevant Chunks

In [ ]:
def retrieve(query, vector_store, embedder, top_k=3):
    """Embed a query and retrieve the top-k most relevant chunks."""
    query_embedding = embedder.encode(query, convert_to_numpy=True)
    results = vector_store.search(query_embedding, top_k=top_k)
    return results


# Test the retriever
test_query = "How does BERT use masked language modeling?"
results = retrieve(test_query, vector_store, embedder, top_k=3)

print(f'Query: "{test_query}"')
print()
for rank, r in enumerate(results, 1):
    print(f'Rank #{rank} — Chunk {r["chunk_id"]} — Score: {r["score"]:.4f}')
    print(textwrap.fill(r['text'], width=90, initial_indent='  ', subsequent_indent='  '))
    print()

## Step 7: The Generator — Load Flan-T5

`google/flan-t5-small` is an instruction-tuned model that runs on CPU.
It won't produce GPT-4 quality answers, but it demonstrates the pipeline clearly.

In [ ]:
print('Loading Flan-T5 generator model (this may take a minute)...')
gen_model_name = 'google/flan-t5-small'
gen_tokenizer = T5Tokenizer.from_pretrained(gen_model_name)
gen_model = T5ForConditionalGeneration.from_pretrained(gen_model_name).to(device)
gen_model.eval()

print(f'Generator loaded: {sum(p.numel() for p in gen_model.parameters()):,} parameters')

In [ ]:
def generate_answer(question, context_chunks, gen_model, gen_tokenizer, max_new_tokens=200):
    """
    Build a prompt with retrieved context and generate an answer.
    
    This is exactly what RAG does:
    1. Concatenate the top-k retrieved chunks as context
    2. Add the question
    3. Ask the LLM to answer based on context only
    """
    context_text = '\n\n'.join([f'[Chunk {i+1}]: {r["text"]}'
                                 for i, r in enumerate(context_chunks)])
    
    prompt = f"""Answer the question based only on the provided context.

Context:
{context_text}

Question: {question}

Answer:"""
    
    # Tokenize
    inputs = gen_tokenizer(
        prompt,
        return_tensors='pt',
        max_length=1024,
        truncation=True
    ).to(device)

    # Generate
    with torch.no_grad():
        output_ids = gen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True
        )

    answer = gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return prompt, answer

## Step 8: Full RAG Pipeline — Ask Questions!

In [ ]:
def rag_query(question, vector_store, embedder, gen_model, gen_tokenizer, top_k=3, verbose=True):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks
    2. Build prompt with context
    3. Generate answer
    """
    print(f'Question: {question}')
    print('─' * 70)

    # Step 1: Retrieve
    retrieved = retrieve(question, vector_store, embedder, top_k=top_k)

    if verbose:
        print(f'Retrieved {len(retrieved)} chunks:')
        for r in retrieved:
            print(f'  [Chunk {r["chunk_id"]}, score={r["score"]:.4f}] {r["text"][:80]}...')
        print()

    # Step 2 & 3: Build prompt + Generate
    prompt, answer = generate_answer(question, retrieved, gen_model, gen_tokenizer)

    print(f'Answer: {answer}')
    print()
    return answer


# --- Ask questions! ---
questions = [
    "What are the two pre-training tasks used in BERT?",
    "How many parameters does BERT-base have?",
    "What is the difference between BERT and SBERT?",
    "How does RAG use cosine similarity?",
    "What is the limitation of Word2Vec compared to BERT?",
]

for q in questions:
    rag_query(q, vector_store, embedder, gen_model, gen_tokenizer, top_k=2, verbose=True)
    print('=' * 70)
    print()

## Step 9: Inspect the Full Prompt

Let's see exactly what gets sent to the generator — this is the heart of RAG.

In [ ]:
question = "What is masked language modeling?"
retrieved = retrieve(question, vector_store, embedder, top_k=2)
prompt, answer = generate_answer(question, retrieved, gen_model, gen_tokenizer)

print('FULL PROMPT SENT TO GENERATOR:')
print('=' * 70)
print(prompt)
print('=' * 70)
print(f'\nGENERATED ANSWER: {answer}')

## Step 10: What Happens Without RAG?

Let's ask the same question directly without any context to see why RAG matters.

In [ ]:
def generate_no_rag(question, gen_model, gen_tokenizer):
    """Ask the model directly, without any retrieved context."""
    prompt = f"Question: {question}\nAnswer:"
    inputs = gen_tokenizer(prompt, return_tensors='pt', max_length=256, truncation=True).to(device)
    with torch.no_grad():
        output_ids = gen_model.generate(**inputs, max_new_tokens=150, num_beams=4)
    return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True)


# Compare RAG vs no-RAG
test_question = "How many parameters does BERT-base have?"

print(f'Question: "{test_question}"')
print()

# Without RAG
no_rag_answer = generate_no_rag(test_question, gen_model, gen_tokenizer)
print(f'WITHOUT RAG: {no_rag_answer}')
print()

# With RAG
retrieved = retrieve(test_question, vector_store, embedder, top_k=2)
_, rag_answer = generate_answer(test_question, retrieved, gen_model, gen_tokenizer)
print(f'WITH RAG:    {rag_answer}')
print()
print('RAG grounds the answer in your document — even a small model can be accurate!')

## Step 11: Try Your Own Document!

Replace `MY_DOCUMENT` with any text you want to query, and ask your own questions.

In [ ]:
MY_DOCUMENT = """
Paste your own document here!
It can be any text — a news article, a research paper summary,
company policy documents, or even notes from a meeting.

The RAG pipeline will chunk it, embed it, and let you ask
natural language questions about it.
"""

MY_QUESTION = "What is this document about?"

# Build index from your document
my_chunks = chunk_text(MY_DOCUMENT, chunk_size=100, overlap=20)
my_embeddings = embedder.encode(my_chunks, convert_to_numpy=True)

my_store = NumpyVectorStore()
my_store.add(my_chunks, my_embeddings)

# Ask your question
rag_query(MY_QUESTION, my_store, embedder, gen_model, gen_tokenizer, top_k=2)

## Key Takeaways

You just built a complete RAG system from scratch! Here's what each component does:

| Component | What it does | Real-world equivalent |
|---|---|---|
| **Chunker** | Splits doc into overlapping windows | LangChain `RecursiveCharacterTextSplitter` |
| **Embedder** | SBERT → fixed vector per chunk | OpenAI `text-embedding-ada-002`, BAAI/bge |
| **Vector Store** | Stores & searches by cosine sim | FAISS, Pinecone, Chroma, Weaviate |
| **Retriever** | embed query → find top-k chunks | LangChain retriever |
| **Generator** | Flan-T5 reads context + answers | GPT-4, Claude, Llama 3 |

**Next steps:**
- Swap `NumpyVectorStore` with FAISS for million-scale search
- Swap Flan-T5 with a larger model via Ollama (local) or OpenAI API
- Add metadata filtering: only retrieve from specific doc sections
- Experiment with chunk sizes and overlap — they affect quality a lot!

**Congratulations — you've gone from Word2Vec → BERT → SBERT → RAG!** 🎉